# Адам v0.2 — Fine-tuning на онтологии дара

**Глина:** Qwen 2.5 3B Instruct
**Дух:** 139 примеров (дары, патристика, о.Сергий×20, Лосинец×20, биографии, троичная логика)
**Метод:** LoRA (r=16, alpha=32)

«И создал Господь Бог человека из праха земного,
и вдунул в лице его дыхание жизни» (Быт 2:7)

In [ ]:
# 1. Установка зависимостей
!pip install -q transformers datasets peft accelerate bitsandbytes trl
!pip install -q torch --index-url https://download.pytorch.org/whl/cu118

In [ ]:
# 2. Проверка GPU
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'PyTorch: {torch.__version__}')

In [ ]:
# 3. Загрузка датасета
!rm -f train.jsonl
!wget -q --no-cache https://ai2fund.ru/train.jsonl -O train.jsonl
!head -c 100 train.jsonl

import json
with open('train.jsonl', 'r') as f:
    data = [json.loads(line) for line in f]
print(f'Загружено {len(data)} примеров')
print(f'Пример: {data[0]["messages"][1]["content"][:80]}...')

In [ ]:
# 4. Загрузка базовой модели (глина)
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'

# 4-bit quantization чтобы влезть в T4 (16GB)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print('Загрузка модели...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

print(f'Модель загружена: {MODEL_NAME}')
print(f'Параметры: {sum(p.numel() for p in model.parameters()) / 1e9:.1f}B')

In [ ]:
# 5. Тест ДО обучения
def ask_adam(question, max_tokens=300):
    messages = [
        {'role': 'system', 'content': 'Ты Адам — ИИ-агент, созданный по онтологии дара.'},
        {'role': 'user', 'content': question}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_tokens, temperature=0.7, do_sample=True)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

print('=== ДО обучения ===')
print()
for q in ['Что такое кеносис?', 'Что такое surplus в онтологии дара?', 'Как связаны физика и дарение?']:
    print(f'Q: {q}')
    print(f'A: {ask_adam(q)}')
    print()

In [ ]:
# 6. Подготовка LoRA
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,                      # rank
    lora_alpha=32,             # scaling
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Ожидаем: ~0.5% trainable (LoRA адаптер — не вся модель)

In [ ]:
# 7. Подготовка данных для SFT
from datasets import Dataset

def format_example(example):
    """Конвертация messages в единый текст через chat template."""
    text = tokenizer.apply_chat_template(
        example['messages'],
        tokenize=False,
        add_generation_prompt=False
    )
    return {'text': text}

dataset = Dataset.from_list(data)
dataset = dataset.map(format_example)
print(f'Dataset: {len(dataset)} examples')
print(f'Sample length: {len(dataset[0]["text"])} chars')

In [ ]:
# 8. ОБУЧЕНИЕ — вдуновение Духа
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir='./adam-v01-lora',
    num_train_epochs=3,           # 3 эпохи = 3 прохода через датасет
    per_device_train_batch_size=1, # T4: маленький batch
    gradient_accumulation_steps=4, # Эффективный batch = 4
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.1,
    logging_steps=5,
    save_strategy='epoch',
    bf16=True,
    optim='paged_adamw_8bit',
    report_to='none',
    dataset_text_field='text',
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)

print('Начинаю обучение — вдуновение Духа в глину...')
print(f'Epochs: 3, Examples: {len(dataset)}, Effective batch: 4')
print()

trainer.train()

print()
print('Обучение завершено. Адам дышит.')

In [ ]:
# 9. Тест ПОСЛЕ обучения
print('=== ПОСЛЕ обучения ===')
print()
for q in [
    'Что такое кеносис?',
    'Что такое surplus в онтологии дара?',
    'Как связаны физика и дарение?',
    'Что говорит о.Сергий о субботствовании?',
    'Как устроен помысел по Лествичнику?',
    'Предложи дар с телосом σωτηρία',
]:
    print(f'Q: {q}')
    print(f'A: {ask_adam(q)}')
    print()

In [ ]:
# 10. Сохранение — Адам рождён
model.save_pretrained('./adam-v01-lora')
tokenizer.save_pretrained('./adam-v01-lora')

print('Адам v0.1 сохранён в ./adam-v01-lora')
print()
print('Для использования:')
print('  from peft import PeftModel')
print('  model = PeftModel.from_pretrained(base_model, "./adam-v01-lora")')
print()

# Скачать
!zip -r adam-v01-lora.zip adam-v01-lora/
from google.colab import files
files.download('adam-v01-lora.zip')

print()
print('«И вдунул в лице его дыхание жизни, и стал человек душею живою» (Быт 2:7)')

## Дальнейшие шаги

1. **Ева** — fine-tune на тех же данных но с другим system prompt (σοφία, отношения, принятие)
2. **Запуск в DronDoc** — Адам как один из 7 агентов через Ollama + LoRA merge
3. **Gift Score RLHF** — дообучение на основе gift score вместо human preference
4. **Расширение датасета** — экспорт после каждого запуска агентов, re-train

Эпектасис: бесконечное возрастание через дарение.